In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# 初始账户状态
target_data = [
    (1001, 1000),
    (1002, 2000)
]

target_columns = ["account_id", "balance"]

target_df = spark.createDataFrame(target_data, target_columns)

# 写入 Delta 表
target_df.write.format("delta").mode("overwrite").saveAsTable("bank_accounts_cdc")

spark.sql("SELECT * FROM bank_accounts_cdc").show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|   1000|
|      1002|   2000|
+----------+-------+



In [0]:
cdc_data = [
    (1001, 1500, "UPDATE"),
    (1002, None, "DELETE"),
    (1003, 3000, "INSERT")
]

cdc_columns = ["account_id", "balance", "change_type"]

cdc_df = spark.createDataFrame(cdc_data, cdc_columns)

cdc_df.createOrReplaceTempView("cdc_source")

cdc_df.show()

+----------+-------+-----------+
|account_id|balance|change_type|
+----------+-------+-----------+
|      1001|   1500|     UPDATE|
|      1002|   NULL|     DELETE|
|      1003|   3000|     INSERT|
+----------+-------+-----------+



In [0]:
%sql
MERGE INTO bank_accounts_cdc AS target
USING cdc_source AS source
ON target.account_id = source.account_id

WHEN MATCHED AND source.change_type = 'DELETE'
  THEN DELETE

WHEN MATCHED AND source.change_type = 'UPDATE'
  THEN UPDATE SET balance = source.balance

WHEN NOT MATCHED AND source.change_type = 'INSERT'
  THEN INSERT (account_id, balance)
       VALUES (source.account_id, source.balance)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,1,1


In [0]:
spark.sql("SELECT * FROM bank_accounts_cdc").show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|   1500|
|      1003|   3000|
+----------+-------+

